The outline for the simulation was created by Yves Heri, while the implementation was made by Owen Holleman as part of the onboarding process for the Plasma, Beams, and Interface Science Group at the University of Michigan.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

In [ ]:
#Functions needed for simulation of Poisson's equation
#Define Relevant Global Variables 
m_e = 9.109e-31
e = 1.6e-19
epsilon_0 = 8.854e-12

#Avoids a divide-by-0 error while solving Poisson's Equation
epsilon = 1e-6

#Define Variables for Euler's Equations
stepCount = 500000

#Poisson Equation takes in the state and returns the rate of change for the Electric Field and Electric Potential
def poissonEQ(f, J):
    phi, E = f   
    dphidx = -E                                          #Since del(phi) = - E for a 1D system we get dphidx=-E
    dEdx = -(J/epsilon_0) * np.sqrt(m_e/(2*e*phi))       #Derived from e(phi) = (mv^2)/2; J = -rho(v) for electrons; and del^2 phi = rho/eps 
    return [dphidx, dEdx]

#Euler's method as a function
def euler(y, y_prime, h):
    return y + h * y_prime

#Combines the euler method with the values of dE and dphi in order to get the phi(d)
def poissonSolver(J, d):
    h=d/stepCount
    phi_func = [epsilon]
    E_func = [0]  
    for i in range(1, stepCount):
        f = [phi_func[-1], E_func[-1]]
        sol = poissonEQ(f, J)
        phi = euler(f[0], sol[0], h)
        E = euler(f[1], sol[1], h)

        phi_func.append(phi)
        E_func.append(E)
    phi_d = phi_func[-1]
    return phi_d

In [ ]:
#Functions for finding max current using the above functions

#Since current is sweeped along orders of magnitudes we find the middle of the log 
# of the two bounds to then use the bisection method with
def logMid(low, high):
    logLow = np.log(low)
    logHigh = np.log(high)
    logAvg = (logLow + logHigh)/2
    return np.exp(logAvg)

#Uses the bisection method to iterate through J in order to find where phi(d) = V0
def maxCurrent(V0, d):
    JLow = 1
    JHigh = 1e10
    phi_d = 0
    while np.abs(phi_d - V0) > 0.1:
        JMid = logMid(JLow, JHigh)
        phi_d = poissonSolver(JMid, d)
        if(phi_d < V0):
            JLow = JMid
        else:
            JHigh = JMid
    return JMid

#Conducts a Sweep of an input coupled with a constant to find maximum current
#param defines what is the input and for param = v voltage is sweeped otherwise distance will be sweeped
def currentSweep(inputSweep, constant, param):
    output = []
    if param == 'v':
        for i in inputSweep:
            Jnow = maxCurrent(i, constant)
            output.append(Jnow)
    else:
        for i in inputSweep:
            Jnow = maxCurrent(constant, i)
            output.append(Jnow)
    output = np.round(output)
    return output


In [ ]:
#Doing sweeps of V0 and d while keeping the other constant
#Sets up variables for vsweep
VSweep = np.linspace(100, 10000, 20)
d = 1e-3

#Sweeps V and generates a J for each V
J4VSweep = currentSweep(VSweep, d, param = 'v')

#Sets up variables for dsweep
dSweep = np.linspace(1e-4, 1e-2, 20)
V0 = 1e3

#Sweeps d and generates a J for each d
J4dSweep = currentSweep(dSweep, V0, param = 'd')

#Plotting the sweeps
fig, (ax1, ax2) = plt.subplots(1,2)
fig.suptitle('V and D sweeps vs Maximum Current Density')

#V0 vs J plot   
ax1.plot(VSweep, J4VSweep, color = 'red')

ax1.set_xlabel("Potential Difference (V)")
ax1.set_ylabel("Maximum Current Density (A/m^2)")

#d vs J plot
ax2.plot(dSweep, J4dSweep, color = 'blue')

ax2.set_xlabel("Capacitor Distance (m)")